# Continuous-focusing lattice 

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize

from orbit.core.bunch import Bunch
from orbit.core.bunch import BunchTwissAnalysis
from orbit.core.spacecharge import SpaceChargeCalc2p5D
from orbit.bunch_utils import collect_bunch
from orbit.envelope import Envelope
from orbit.envelope import EnvelopeTracker
from orbit.space_charge.sc2p5d import SC2p5D_AccNode
from orbit.space_charge.sc2p5d import setSC2p5DAccNodes
from orbit.teapot import ContinuousLinearFocusingTEAPOT
from orbit.teapot import TEAPOT_Lattice
from orbit.utils.consts import mass_proton

from utils import track_env_tbt
from utils import track_bunch_tbt
from utils import samp_dist

plt.style.use("../style.mplstyle")

## Matched envelope calculation

In [ ]:
def calc_eq_radius(emittance: float, omega: float) -> float:
    return np.sqrt(4.0 * emittance / omega)

def calc_depressed_freq(perveance: float, omega_0: float, radius: float, pad: float = 1e-12) -> float:
    return np.sqrt(omega_0**2 - perveance / (radius**2 + pad))

In [ ]:
class EquilibriumParameters:
    def __init__(self, length: float, sigma_0: float, sigma: float, emittance: float) -> None:
        self.length = length
        self.emittance = emittance

        self.sigma_0 = sigma_0
        self.omega_0 = sigma_0 * length
        self.sigma = None
        self.omega = None
                
        self.perveance = None
        self.radius = None
        self.set_sigma(sigma)
    
    def set_sigma(self, sigma: float) -> float:
        self.sigma = sigma
        self.omega = sigma * self.length        
        self.radius = calc_eq_radius(self.emittance, self.omega)
        self.perveance = (self.omega_0**2 - self.omega**2) * self.radius**2
    
    def set_sigma_0(self, sigma_0: float) -> None:
        self.sigma_0 = sigma_0
        self.omega_0 = sigma_0 * self.length
        self.set_sigma(self.sigma)

    def cov_matrix(self) -> np.ndarray:
        cov_matrix = np.zeros((4, 4))
        cov_matrix[0, 0] = 0.25 * self.radius ** 2
        cov_matrix[2, 2] = cov_matrix[0, 0]
        cov_matrix[1, 1] = self.emittance**2 / cov_matrix[0, 0]
        cov_matrix[3, 3] = cov_matrix[1, 1]
        return cov_matrix

In [ ]:
length = 0.25
sigma_0 = math.radians(80.0 + 360.0)
omega_0 = sigma_0 * length
sigma = 0.5 * sigma_0

emittance = 0.25e-06

eq_params = EquilibriumParameters(
    length=length, 
    sigma_0=sigma_0, 
    sigma=sigma, 
    emittance=emittance,
)

print("sigma =", math.degrees(eq_params.sigma))
print("sigma_0 =", math.degrees(eq_params.sigma_0))
print("sigma / sigma_0 =", eq_params.sigma / eq_params.sigma_0)
print("perveance =", eq_params.perveance)
print("radius =", eq_params.radius)

## Track envelope

In [ ]:
def make_lattice(length: float, sigma_0: float, nparts: int = 20) -> TEAPOT_Lattice:
    omega_0 = sigma_0 * length
    node = ContinuousLinearFocusingTEAPOT(
        length=eq_params.length,
        kq=(omega_0 ** 2),
        nparts=nparts,
    )
    lattice = TEAPOT_Lattice()
    lattice.addNode(node)
    lattice.initialize()
    return lattice

In [ ]:
kin_energy = 0.0025
bunch_length = 10.0

cov_matrix = np.zeros((6, 6))
cov_matrix[0:4, 0:4] = eq_params.cov_matrix() 
cov_matrix[4, 4] = bunch_length**2 / 12.0

bunch = Bunch()
bunch.mass(mass_proton)
bunch.getSyncParticle().kinEnergy(kin_energy)

envelope = Envelope(bunch=bunch, cov_matrix=cov_matrix)
envelope.set_intensity(
    eq_params.perveance 
    * bunch_length
    * (envelope.beta()**2 * envelope.gamma()**3) 
    / (2.0 * envelope.classical_radius)
)

lattice = make_lattice(length=length, sigma_0=sigma_0)

history = track_env_tbt(envelope, lattice, copy=True, turns=100)

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(history["r"])
ax.set_ylim(0.0, np.mean(history["r"]) * 1.8)
ax.set_xlabel("Period")
ax.set_ylabel("Radius")
plt.show()

## Track bunch

In [ ]:
turns = 20
mismatch = 0.0
size = 50_000

particles = np.zeros((size, 6))
particles[:, :4] = samp_dist(
    size=size, 
    name="gauss",
    cov_matrix=envelope.cov_matrix[:4, :4]
)
particles[:, :4] *= 1.0 + np.random.normal(size=(particles.shape[0], 4), scale=0.1)
particles[:, 4] = bunch_length * np.random.uniform(-0.5, 0.5, size=size)

# Mismatch
particles[:, 0] *= 1.0 + mismatch

bunch = Bunch()
bunch.mass(mass_proton)
bunch.getSyncParticle().kinEnergy(kin_energy)
for i in range(particles.shape[0]):
    bunch.addParticle(*particles[i])

bunch.macroSize(envelope.intensity / bunch.getSize())

lattice = make_lattice(length=length, sigma_0=sigma_0, nparts=10)

if envelope.intensity > 0:    
    sc_calc = SpaceChargeCalc2p5D(64, 64, 1)
    sc_path_length_min = 0.001
    sc_nodes = setSC2p5DAccNodes(lattice, sc_path_length_min, sc_calc)

history = track_bunch_tbt(bunch, lattice, copy=False, turns=turns)

fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(history["r"])
ax.set_ylim(0.0, np.mean(history["r"]) * 1.8)
ax.set_xlabel("Period")
ax.set_ylabel("Radius")
plt.show()

In [ ]:
particles = collect_bunch(bunch)["coords"]
particles = np.copy(particles)
particles = particles[:, :4]
particles = 1000.0 * particles

axis = (0, 1)
xmax = 1.0 * np.max(np.abs(particles), axis=0)
xmax = 4.0 * np.std(particles, axis=0)
limits = list(zip(-xmax, xmax))

hist, edges = np.histogramdd(
    particles[:, axis], 
    bins=50, 
    range=[limits[k] for k in axis]
)
hist = hist / np.max(hist)
hist = np.ma.masked_less_equal(hist, 0.0)
log_hist = np.ma.log10(hist)

fig, axs = plt.subplots(ncols=3, sharex=True, sharey=True, figsize=(8, 2.5))
axs[0].pcolormesh(edges[0], edges[1], hist.T, cmap="Blues")
axs[1].pcolormesh(edges[0], edges[1], log_hist.T, vmin=-4.0, cmap="Blues")
axs[2].scatter(
    particles[:10_000, axis[0]],
    particles[:10_000, axis[1]], 
    c="black", 
    ec="none", 
    s=2
)
for ax in axs:
    ax.set_xlim(limits[axis[0]])
    ax.set_ylim(limits[axis[1]])
plt.show()